In [46]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob

def analyze_hyperparameters(directory_path, plot_show=True, threshold=0.5):
    """
    Searches for a single CSV file in the given directory, performs analysis,
    generates visualizations, and saves the plots to a dedicated 'analysis_output'
    folder within that same directory.

    Args:
        directory_path (str): The path to the directory containing the CSV file.
        plot_show (bool): If True, displays the plots interactively.
        threshold (float): The value to subtract from the max accuracy to define
                         high-performing models for the summary.
    """
    # --- 1. Find the CSV file ---
    csv_files = glob.glob(os.path.join(directory_path, '*.csv'))

    if len(csv_files) == 1:
        file_path = csv_files[0]
        #print(f"Found CSV file: {file_path}")
    elif len(csv_files) == 0:
        print(f"Error: No CSV files found in '{directory_path}'.")
        return
    else:
        print(f"Error: More than one CSV file found in '{directory_path}'. Please ensure there is only one.")
        print("Found files:", csv_files)
        return

    # --- 2. Define and Create Output Directory ---
    output_dir = os.path.join(directory_path, 'analysis_output')
    os.makedirs(output_dir, exist_ok=True)
    print(f"\nOutput will be saved to: {output_dir}")

    # --- 3. Load Data ---
    try:
        df = pd.read_csv(file_path)
        #print(f"Successfully loaded data from '{file_path}'")
    except FileNotFoundError:
        # This is a fallback, though the glob check should prevent it.
        print(f"Error: The file '{file_path}' was not found.")
        return

    # --- 4. Correlation Analysis ---
    columns_to_correlate = ['gap', 'seq_num', 'hidden_size', 'num_layer', 'ACC']
    corr_matrix = df[columns_to_correlate].corr(method='spearman')

    plt.figure(figsize=(10, 8))
    sns.heatmap(
        corr_matrix,
        annot=True,
        cmap='coolwarm',
        fmt=".2f",
        linewidths=.5
    )
    plt.title('Spearman Correlation Matrix of Parameters and Accuracy', fontsize=16)

    corr_save_path = os.path.join(output_dir, 'correlation_matrix.png')
    plt.savefig(corr_save_path, dpi=300, bbox_inches='tight')
    #print(f"Correlation matrix saved to '{corr_save_path}'")
    if plot_show:
        plt.show()
    else:
        plt.close()

    # --- 5. Visualize Impact of Window Size (seq_num) ---
    fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(20, 14))
    fig.suptitle('Impact of Window Size on Accuracy', fontsize=20)

    boxplot_props = {
        "showmeans": True,
        "meanprops": {
            "marker": "o",
            "markerfacecolor": "white",
            "markeredgecolor": "black",
            "markersize": "8"
        }
    }

    # Plot 1: Overall impact
    sns.boxplot(ax=axes[0, 0], x='seq_num', y='ACC', data=df, **boxplot_props)
    axes[0, 0].set_title('Overall Distribution', fontsize=16)
    axes[0, 0].set_xlabel('Window Size', fontsize=14)
    axes[0, 0].set_ylabel('Accuracy (%)', fontsize=14)
    axes[0, 0].grid(True, axis='y')

    # Plot 2: Grouped by Gap
    sns.boxplot(ax=axes[0, 1], x='seq_num', y='ACC', hue='gap', data=df, **boxplot_props)
    axes[0, 1].set_title('Grouped by Gap', fontsize=16)
    axes[0, 1].set_xlabel('Window Size', fontsize=14)
    axes[0, 1].set_ylabel('')
    axes[0, 1].grid(True, axis='y')
    axes[0, 1].legend(title='Gap')

    # Plot 3: Grouped by Hidden Size
    sns.boxplot(ax=axes[1, 0], x='seq_num', y='ACC', hue='hidden_size', data=df, **boxplot_props)
    axes[1, 0].set_title('Grouped by Hidden Size', fontsize=16)
    axes[1, 0].set_xlabel('Window Size', fontsize=14)
    axes[1, 0].set_ylabel('Accuracy (%)', fontsize=14)
    axes[1, 0].grid(True, axis='y')
    axes[1, 0].legend(title='Hidden Size')

    # Plot 4: Grouped by Number of Layers
    sns.boxplot(ax=axes[1, 1], x='seq_num', y='ACC', hue='num_layer', data=df, **boxplot_props)
    axes[1, 1].set_title('Grouped by Number of Layers', fontsize=16)
    axes[1, 1].set_xlabel('Window Size', fontsize=14)
    axes[1, 1].set_ylabel('')
    axes[1, 1].grid(True, axis='y')
    axes[1, 1].legend(title='Num Layers')

    for ax in axes.flatten():
        ax.tick_params(axis='both', which='major', labelsize=12)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])

    impact_save_path = os.path.join(output_dir, 'window_size_impact.png')
    plt.savefig(impact_save_path, dpi=300, bbox_inches='tight')
    #print(f"Window size impact plot saved to '{impact_save_path}'")
    if plot_show:
        plt.show()
    else:
        plt.close()


    # --- 6. Summary Statistics ---
    high_accuracy_threshold = df.ACC.max() - threshold
    num_high_performers = df.loc[df.ACC > high_accuracy_threshold].shape[0]
    print(f"{num_high_performers} models reached accuracy over {high_accuracy_threshold:.2f}%. Highest Accuracy Recorded: {df.ACC.max():.4f}%")

    print(df.loc[df.ACC > high_accuracy_threshold])



# Source Robot: Franka_main

In [47]:
analyze_hyperparameters(directory_path='trained_models/franka_main/contact_detection_v2', plot_show= False, threshold=0.05)
analyze_hyperparameters(directory_path='trained_models/franka_main/contact_detection_v3/64', plot_show= False, threshold=0.2)


Output will be saved to: trained_models/franka_main/contact_detection_v2/analysis_output
5 models reached accuracy over 95.90%. Highest Accuracy Recorded: 95.9478%
     gap  seq_num  hidden_size  num_layer        ACC
69     3      200          256          1  95.942538
296    3      250          128          3  95.908482
309    3      300          256          1  95.947764
311    3      300          256          3  95.913747
419    3      200         1024          3  95.913005

Output will be saved to: trained_models/franka_main/contact_detection_v3/64/analysis_output
5 models reached accuracy over 92.63%. Highest Accuracy Recorded: 92.8317%
     gap  seq_num  hidden_size  num_layer        ACC  train_size  val_size
150    3      100          128          1  92.659857      264232    158689
155    3      100          256          3  92.673720      264232    158689
209    5      150           64          3  92.650265      153625    153679
240    3      200           32          1  92.831

In [108]:
analyze_hyperparameters(directory_path='trained_models/franka_main/contact_localization_v2', plot_show= False, threshold=0.08)
analyze_hyperparameters(directory_path='trained_models/franka_main/contact_localization_v3/67', plot_show= False, threshold=0.2)


Output will be saved to: trained_models/franka_main/contact_localization_v2/analysis_output
3 models reached accuracy over 98.67%. Highest Accuracy Recorded: 98.7460%
    gap  seq_num  hidden_size  num_layer        ACC
57    3      150          256          1  98.745965
59    3      150          256          3  98.717794
69    3      200          256          1  98.678003

Output will be saved to: trained_models/franka_main/contact_localization_v3/67/analysis_output
3 models reached accuracy over 92.35%. Highest Accuracy Recorded: 92.5495%
     seq_num  gap  hidden_size  num_layer        ACC  train_size  val_size
150      150    3          128          1  92.351070      112312     73239
187      200    3          128          2  92.549460      112220     70411
191      200    3          256          3  92.381872      112220     70411


# Target Robot: Franka_Mindlab

In [105]:
analyze_hyperparameters(directory_path='trained_models/franka_mindlab/contact_detection_v2', plot_show= False, threshold=0.02)
analyze_hyperparameters(directory_path='trained_models/franka_mindlab/contact_detection_v3/64', plot_show= False, threshold=0.2)


Output will be saved to: trained_models/franka_mindlab/contact_detection_v2/analysis_output
2 models reached accuracy over 96.70%. Highest Accuracy Recorded: 96.7215%
     gap  seq_num  hidden_size  num_layer        ACC
90     3      200           32          1  96.721549
105    3      200         1024          1  96.714296

Output will be saved to: trained_models/franka_mindlab/contact_detection_v3/64/analysis_output
2 models reached accuracy over 88.61%. Highest Accuracy Recorded: 88.8132%
     gap  seq_num  hidden_size  num_layer        ACC  train_size  val_size
154    3      100          256          2  88.662159       13792      8282
217   10      150           32          2  88.813181        4041      8072


In [104]:
analyze_hyperparameters(directory_path='trained_models/franka_mindlab/contact_localization_v2', plot_show= False, threshold=0.03)
analyze_hyperparameters(directory_path='trained_models/franka_mindlab/contact_localization_v3', plot_show= False, threshold=0.08)


Output will be saved to: trained_models/franka_mindlab/contact_localization_v2/analysis_output
3 models reached accuracy over 99.85%. Highest Accuracy Recorded: 99.8796%
     gap  seq_num  hidden_size  num_layer        ACC  train_size_localization  \
135    3      300          256          1  99.879639                     4984   
138    3      300          512          1  99.859579                     4984   
139    3      300          512          2  99.859579                     4984   

     train_size_detection  
135                 26874  
138                 26874  
139                 26874  

Output will be saved to: trained_models/franka_mindlab/contact_localization_v3/analysis_output
2 models reached accuracy over 87.58%. Highest Accuracy Recorded: 87.6638%
     seq_num  gap  hidden_size  num_layer        ACC  train_size  val_size
257      300    3           64          3  87.625289        4598      2594
272      300    5          128          3  87.663840        2760      2

# Target Robot: UR5

In [100]:
analyze_hyperparameters(directory_path='trained_models/ur5/contact_detection_v2', plot_show= False, threshold=0.2)
analyze_hyperparameters(directory_path='trained_models/ur5/contact_detection_v3/64/', plot_show= False, threshold=0.08)


Output will be saved to: trained_models/ur5/contact_detection_v2/analysis_output
3 models reached accuracy over 94.14%. Highest Accuracy Recorded: 94.3411%
     gap  seq_num  hidden_size  num_layer        ACC
122    3      250          512          3  94.185308
126    3      300           32          1  94.341075
130    3      300           64          2  94.155011

Output will be saved to: trained_models/ur5/contact_detection_v3/64/analysis_output
2 models reached accuracy over 84.24%. Highest Accuracy Recorded: 84.3187%
     gap  seq_num  hidden_size  num_layer        ACC  train_size  val_size
117    5       80          256          1  84.311278        9607      9612
155    3      100          256          3  84.318658       15888      9540


In [96]:
analyze_hyperparameters(directory_path='trained_models/ur5/contact_localization_v2', plot_show= False, threshold=0.04)
analyze_hyperparameters(directory_path='trained_models/ur5/contact_localization_v3', plot_show= False, threshold=0.4)


Output will be saved to: trained_models/ur5/contact_localization_v2/analysis_output
2 models reached accuracy over 99.23%. Highest Accuracy Recorded: 99.2718%
     gap  seq_num  hidden_size  num_layer        ACC  train_size_localization  \
132    3      300          128          1  99.247573                     4119   
134    3      300          128          3  99.271845                     4119   

     train_size_detection  
132                 31172  
134                 31172  

Output will be saved to: trained_models/ur5/contact_localization_v3/analysis_output
2 models reached accuracy over 86.25%. Highest Accuracy Recorded: 86.6473%
     seq_num  gap  hidden_size  num_layer        ACC  train_size  val_size
353      400   10           64          3  86.510129        1080      2172
379      450    5          128          2  86.647315        2155      2067


# testing


In [110]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import os
import sys
import logging
import time
import glob
from collections import Counter
from torch.utils.data import DataLoader

# --- Add project root to path ---
project_root = os.getcwd().replace('pipelines', '')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Make sure the necessary modules are accessible
# NOTE: Update this path if your model file is located elsewhere
from models.transformer_contactDetection import TransformerModel 
from src_v3.dataset_loader import LoadSeqDataset

# --- 1. Helper Functions ---
def majority_voting_last_n(model_out, n):
    """
    Applies a majority voting filter over the last n predictions to smooth the output.
    """
    model_out_series = pd.Series(model_out)
    smoothed_predictions = model_out_series.copy()
    
    # Calculate rolling window majority
    rolling_window = model_out_series.rolling(window=n, min_periods=1)
    # Use a lambda to find the most common element in the window
    smoothed_predictions = rolling_window.apply(lambda x: Counter(x).most_common(1)[0][0], raw=False).astype(int)
    
    return smoothed_predictions.tolist()

def contact_detection_accuracy(df):
    """
    Calculates TP, TN, FP, FN, and contact delays from a dataframe of results.
    The delay is now calculated in seconds.
    """
    TP, TN, FP, FN = 0, 0, 0, 0
    contact_delays_seconds = []
    df = df.reset_index(drop=True)

    df['label_diff'] = df['label'].diff().fillna(0)
    change_events = df[df['label_diff'] != 0].index.tolist()

    if 0 not in change_events:
        change_events.insert(0, 0)
    if len(df) - 1 not in change_events:
        change_events.append(len(df))

    for i in range(len(change_events) - 1):
        start_idx = change_events[i]
        end_idx = change_events[i+1]
        segment = df.iloc[start_idx:end_idx]
        
        if segment.empty:
            continue

        is_contact_segment = segment['label'].iloc[0] == 1

        if is_contact_segment:
            TP += (segment['majority_voting'] == 1).sum()
            FN += (segment['majority_voting'] == 0).sum()
            
            first_detection = segment[segment['majority_voting'] == 1]
            if not first_detection.empty:
                # --- METRIC CALCULATION IN SECONDS ---
                first_detection_time = first_detection.iloc[0]['time']
                segment_start_time = segment.iloc[0]['time']
                delay = first_detection_time - segment_start_time
                contact_delays_seconds.append(delay)
                # --- END OF CHANGE ---
        else: # No-contact segment
            TN += (segment['majority_voting'] == 0).sum()
            FP += (segment['majority_voting'] == 1).sum()

    return TP, TN, FP, FN, contact_delays_seconds

# --- 2. Main Evaluation Function ---
def evaluate_specific_transformer_model():
    """
    Loads a specifically named Transformer model and evaluates its performance.
    """
    # --- Configuration ---
    data_name = 'franka_main'
    dof = 7
    batch_size = 1024
    n_majority_voting = 15

    # --- Specify the model to evaluate ---
    model_dir = os.path.join(project_root, 'pipelines', 'trained_models', 'franka_main', 'contact_detection_transformer', '64')
    model_name_part = "d_model128_n_head4_numLayer2_seq_num300_gap1"
    
    search_pattern = os.path.join(model_dir, f"{model_name_part}_accuracy*.pth")
    #search_pattern = os.path.join(model_dir, f"{model_name_part}.pth")

    model_paths = glob.glob(search_pattern)

    if not model_paths:
        logging.error(f"No model file found matching the pattern: {search_pattern}")
        return
    
    model_path = model_paths[0]
    logging.info(f"Evaluating specific model: {os.path.basename(model_path)}")

    # --- Parse Hyperparameters from Filename ---
    parts = os.path.basename(model_path).replace('.pth', '').split('_')
    hyperparams = {}
    try:
        for part in parts:
            if 'model' in part: hyperparams['d_model'] = int(part.replace('model', ''))
            elif 'head' in part: hyperparams['n_head'] = int(part.replace('head', ''))
            elif 'numLayer' in part: hyperparams['num_layer'] = int(part.replace('numLayer', ''))
            elif 'num' in part: hyperparams['seq_num'] = int(part.replace('num', ''))
    except (ValueError, IndexError):
        logging.error(f"Could not parse hyperparameters from filename: {os.path.basename(model_path)}")
        return
    
    # --- Load Model ---
    model = TransformerModel(
        num_features=hyperparams['seq_num'],
        d_model=hyperparams['d_model'],
        nhead=hyperparams['n_head'],
        dim_feedforward=hyperparams['d_model']*4,
        num_encoder_layers=hyperparams['num_layer']
    )
    
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()

    # --- Data Loading and Evaluation Loop ---
    data_directory = os.path.join(project_root, 'dataset', data_name, 'labeled_data')
    selected_features = [f'e{i}' for i in range(dof)]
    all_csv_files = glob.glob(os.path.join(data_directory, '**', '*.csv'), recursive=True)

    total_TP, total_TN, total_FP, total_FN = 0, 0, 0, 0
    all_contact_delays = []
    per_file_results = []

    for file_path in all_csv_files:
        label_val = 1 if 'no_contact' not in file_path else 0
        
        trial_dataset = LoadSeqDataset(file_path, torch.tensor(label_val), selected_features, seq_num=hyperparams['seq_num'], gap=1, mode='val')
        if len(trial_dataset) == 0:
            continue
            
        test_loader = DataLoader(trial_dataset, batch_size=batch_size, shuffle=False)
        
        predictions = []
        with torch.no_grad():
            for inputs, _ in test_loader:
                inputs = inputs.to(device)
                outputs = (torch.sigmoid(model(inputs)) > 0.5).int()
                predictions.extend(outputs.cpu().numpy())

        # --- DATAFRAME CREATION WITH TIME ---
        df_results = pd.DataFrame({
            'time': trial_dataset.times, # Include timestamps
            'label': trial_dataset.labels,
            'model_out': predictions
        })
        # --- END OF CHANGE ---
        
        df_results['majority_voting'] = majority_voting_last_n(df_results['model_out'], n_majority_voting)

        TP, TN, FP, FN, contact_delays = contact_detection_accuracy(df_results)
        
        file_total = TP + TN + FP + FN
        if file_total > 0:
            file_accuracy = (TP + TN) / file_total * 100
            per_file_results.append({'filename': os.path.basename(file_path), 'accuracy': file_accuracy})
        
        total_TP += TP; total_TN += TN; total_FP += FP; total_FN += FN
        all_contact_delays.extend(contact_delays)

    # --- Display and Save Results ---
    logging.info("\n--- Accuracy Per File ---")
    results_df = pd.DataFrame(per_file_results)
    #logging.info(f"\n{results_df.to_string()}")
    
    output_path = os.path.join(model_dir, 'per_file_accuracy_results.csv')
    results_df.to_csv(output_path, index=False)
    logging.info(f"\nPer-file accuracy results saved to: {output_path}")

    logging.info("\n--- Overall Evaluation Results ---")
    ModelAccuracy = (total_TP + total_TN) / (total_TP + total_TN + total_FP + total_FN) * 100
    DetectionFailureRate = total_FN / (total_TP + total_FN) * 100 if (total_TP + total_FN) > 0 else 0
    FalseAlarmRate = total_FP / (total_TN + total_FP) * 100 if (total_TN + total_FP) > 0 else 0
    
    # --- FINAL METRIC IN SECONDS ---
    avg_contact_delay_seconds = np.nanmean(all_contact_delays) if all_contact_delays else 0
    logging.info(f"Overall Model Accuracy: {ModelAccuracy:.2f}%")
    logging.info(f"Detection Failure Rate: {DetectionFailureRate:.2f}%")
    logging.info(f"False Alarm Rate: {FalseAlarmRate:.2f}%")
    logging.info(f"Average Contact Detection Delay: {avg_contact_delay_seconds:.4f} seconds")
    # --- END OF CHANGE ---

if __name__ == '__main__':
    logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
    evaluate_specific_transformer_model()


2025-10-02 18:16:23,725 - INFO - Evaluating specific model: d_model128_n_head4_numLayer2_seq_num300_gap1_accuracy93.09.pth
2025-10-02 18:17:43,029 - INFO - 
--- Accuracy Per File ---
2025-10-02 18:17:43,033 - INFO - 
Per-file accuracy results saved to: /home/rzma/myProjects/contactInterpretation/pipelines/trained_models/franka_main/contact_detection_transformer/64/per_file_accuracy_results.csv
2025-10-02 18:17:43,034 - INFO - 
--- Overall Evaluation Results ---
2025-10-02 18:17:43,035 - INFO - Overall Model Accuracy: 89.07%
2025-10-02 18:17:43,035 - INFO - Detection Failure Rate: 13.01%
2025-10-02 18:17:43,035 - INFO - False Alarm Rate: 9.12%
2025-10-02 18:17:43,035 - INFO - Average Contact Detection Delay: 0.0464 seconds
